# Stage-only ATLAS import

Place full TIFF tiles using stage metadata, without cross-correlation or valid-area cropping.
Output TIFFs use lossless Deflate compression. Set `series_folder` below before running.

The workflow skips existing `stage_stitched_S_*.tiff` outputs. These filenames are separate from the raw-import outputs.


In [ ]:
from pathlib import Path
from atlas.io import get_valid_slice_folders

In [ ]:
# Set the dataset folder (relative to the repository root).
series_folder = Path(
    "/home/xcamra/projects/ATLAS/private-data/p09/empty-project_data/"
    "session_769033652/Section Set 1"
)

In [ ]:
series_list, failed_folders = get_valid_slice_folders(series_folder)

In [ ]:
print(f"\nFound {len(series_list)} series.")

for f in series_list:
    print(f)

In [ ]:
import json

import numpy as np
import tifffile as tiff

from atlas.io import extract_s_number, get_metadata_for_stitched_tif
from atlas.stitching import stage_stitching


In [ ]:
# iterate over all folders in the series
for raw_data_folder in series_list:
    # Iterate over all items in the folder
    for file in raw_data_folder.iterdir():  
        # Check if it's a file with the desired extension
        if file.is_file() and file.suffix == ".ve-mif":
            print(f"File with '.ve-mif' extension found: {file.name}")
            mif_file = file
            mif_metadict = get_metadata_for_stitched_tif(mif_file)
            # use naming convention to find the first tif
            first_tif_path = next(raw_data_folder.glob("*.tif"))
            print(first_tif_path)
            extracted_number = extract_s_number(first_tif_path)
            # Define the output file path
            output_tif_path = raw_data_folder.parent.joinpath(f"stage_stitched_S_{extracted_number}.tiff")
            output_cc_path = raw_data_folder.parent.joinpath(f"stage_stitching_S_{extracted_number}.csv")
            output_jason_path = raw_data_folder.parent.joinpath(f"stage_stitched_transforms_S_{extracted_number}.json")

            if output_tif_path.exists():
                print(f"✅ Skipping: {output_tif_path.name} already exists.")
            else:
                print(f"🔄 Stitching image for S_{extracted_number}...")

                try:
                    stitched_img, mif_tile_df, transform_dict = stage_stitching(
                        mif_file,
                    )

                    # Save the stitched image as a TIFF file
                    tiff.imwrite(output_tif_path, 
                                 np.flipud(stitched_img), 
                                 compression="deflate",
                                 metadata={"axes": "YX", "AcquisitionMetadata": mif_metadict})
                    

                    mif_tile_df.to_csv(output_cc_path, index=False)

                    # Convert NumPy arrays to lists for JSON compatibility
                    json_ready_dict = {k: v.tolist() for k, v in transform_dict.items()}

                    # Save to JSON file
                    with open(output_jason_path, "w") as f:
                        json.dump(json_ready_dict, f, indent=2)
                        
                except Exception as e:
                    # Handle any error
                    print(f"Unexpected error with item {file}: {e}")
            

In [ ]:
import tifffile as tiff

In [ ]:
with tiff.TiffFile(series_folder.joinpath("stage_stitched_S_2.tiff")) as tif:
    metadata = tif.shaped_metadata[0]["AcquisitionMetadata"]

for m in metadata.items():
    print(m)